# Pratilipi Reading Recommendation System

## Objective

The goal of this assignment is to build a recommendation system that predicts what a user should read next based on their reading history.

I frame this as a **book-level recommendation problem** supported by chapter-level behavior. The raw interaction data is at chapter level, so I use chapter interactions to infer reader intent, depth of engagement, category preference, and author affinity.

The recommendation system should not treat every reader the same. Instead, it should first understand the reader profile and then decide the most useful next action:

1. Recommend a new book to explore.
2. Recommend a partially explored book worth revisiting.
3. Recommend continuation when chapter-level signals support it.

Since the data does not include timestamps or explicit ratings, I treat interactions as implicit feedback and avoid making claims about exact recency.


## Data-driven problem framing

The assignment mentions chapter progression, and that is important. But before building a sequential recommender, I first check how much progression signal is actually present in the data.

If most user-book combinations contain only one chapter interaction, then a pure next-chapter model would be too narrow. In that case, chapter order is still useful, but mainly as a lightweight engagement-depth signal.

The recommender should therefore use both:

- **book-level behavior**: what books a user has interacted with,
- **chapter-level signals**: how deep into a book the interaction happened,
- **reader profiles**: serious readers, casual readers, category-focused readers, and generalists,
- **book profiles**: popular books, category-trending books, and engagement-quality books.

This keeps the solution aligned with the data rather than forcing a generic recommendation method onto it.


## Reader and book profiling strategy

Rather than treating every user the same, I profile readers based on how they interact with books and categories. This helps the recommender decide the most useful next action: continue reading, revisit a partially explored book, or discover a new book.

### Reader profiles

I create reader segments using interaction frequency, book depth, and tag diversity:

1. **Invested readers**  
   Users with repeated interactions, deeper chapter engagement, or more than one chapter interaction within at least one book.

2. **Category-focused readers**  
   Users whose weighted reading history is concentrated in a small number of tags or genres.

3. **Generalist readers**  
   Users who engage across many different tags without one dominant category.

4. **Casual readers**  
   Users with very limited or shallow interaction history.

### Book profiles

I also profile books because popularity alone is not enough:

1. **Globally popular books**  
   Books with high total interactions or high unique-user counts.

2. **Category-trending books**  
   Books that are popular within their genre/tag groups.

3. **Engagement-quality books**  
   Books with stronger depth signals, such as later chapter interactions or repeated user-book engagement.

4. **Chapter-level opportunity**  
   Chapters with strong interaction volume within a book, useful for deciding whether a user should continue or revisit a book.

### Important limitation

Since timestamps are unavailable, I do not claim true recency or return behavior. Instead, I use interaction frequency, distinct chapters read, chapter depth, author affinity, and tag diversity as proxies for user intent and engagement.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


In [2]:
# The notebook is intended to run from the notebooks/ folder.
# If run from the repo root, the fallback path will still work.

DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")

chapters = pd.read_csv(DATA_DIR / "chapters.csv")
interactions = pd.read_csv(DATA_DIR / "interactions.csv")

print("chapters:", chapters.shape)
print("interactions:", interactions.shape)

display(chapters.head())
display(interactions.head())


chapters: (50000, 6)
interactions: (1000000, 3)


,chapter_id,chapter_sequence_no,book_id,author_id,published_date,tags
0,2812946,1,139726,66847,1990-03-22,Fantasy|Horror
1,4330764,2,139726,66847,1990-04-09,Fantasy|Young Adult|Literary Fiction
2,2664499,3,139726,66847,1990-04-07,Fantasy
3,2260666,4,139726,66847,1990-05-18,Literary Fiction|Fantasy
4,6069976,1,191772,62262,2008-07-30,Horror|Young Adult|Romance|Graphic Novel


,user_id,chapter_id,book_id
0,user_2378720,5894067,444295
1,user_2321122,2532511,785684
2,user_2335775,6777764,999595
3,user_7906001,7366896,748410
4,user_9981689,7853186,418083


In [3]:
print("Chapter columns:", chapters.columns.tolist())
print("Interaction columns:", interactions.columns.tolist())

print("\nMissing values in chapters:")
display(chapters.isna().sum())

print("\nMissing values in interactions:")
display(interactions.isna().sum())

print("\nUnique counts:")
summary = {
    "users": interactions["user_id"].nunique(),
    "books_in_interactions": interactions["book_id"].nunique(),
    "chapters_in_interactions": interactions["chapter_id"].nunique(),
    "books_in_chapters": chapters["book_id"].nunique(),
    "chapters_in_metadata": chapters["chapter_id"].nunique(),
    "authors": chapters["author_id"].nunique(),
}
display(pd.Series(summary))


Chapter columns: ['chapter_id', 'chapter_sequence_no', 'book_id', 'author_id', 'published_date', 'tags']
Interaction columns: ['user_id', 'chapter_id', 'book_id']

Missing values in chapters:


chapter_id             0
chapter_sequence_no    0
book_id                0
author_id              0
published_date         0
tags                   0
dtype: int64


Missing values in interactions:


user_id       0
chapter_id    0
book_id       0
dtype: int64


Unique counts:


users                       149803
books_in_interactions         9575
chapters_in_interactions     49998
books_in_chapters             9575
chapters_in_metadata         50000
authors                       4263
dtype: int64

## Book metadata table

The chapter file is at chapter level, so I first create a book-level metadata table.

For each book, I compute:

- total chapters,
- author,
- earliest published date,
- combined tags.


In [4]:
def combine_tags(series):
    tags = []
    for value in series.dropna().astype(str):
        tags.extend([t.strip() for t in value.split("|") if t.strip()])
    return "|".join(sorted(set(tags)))

book_meta = (
    chapters
    .groupby("book_id")
    .agg(
        total_chapters=("chapter_id", "nunique"),
        author_id=("author_id", "first"),
        first_published_date=("published_date", "min"),
        tags=("tags", combine_tags)
    )
    .reset_index()
)

book_meta["tag_list"] = book_meta["tags"].fillna("").apply(lambda x: [t for t in x.split("|") if t])

display(book_meta.head())
display(book_meta[["total_chapters"]].describe())


,book_id,total_chapters,author_id,first_published_date,tags,tag_list
0,100089,3,55121,1995-03-24,Fantasy|Science Fiction,"[Fantasy, Science Fiction]"
1,100096,1,61876,2015-12-31,Romance,[Romance]
2,100193,4,67983,1996-01-26,Crime|Dystopian|Fantasy|Humor|Literary Fiction...,"[Crime, Dystopian, Fantasy, Humor, Literary Fi..."
3,100205,4,64099,2016-02-26,Adventure|Fantasy|Horror|Literary Fiction|Myst...,"[Adventure, Fantasy, Horror, Literary Fiction,..."
4,100301,3,77538,2015-08-14,Dystopian|Humor,"[Dystopian, Humor]"


,total_chapters
count,9575.000000
mean,5.221932
std,4.056853
min,1.000000
25%,3.000000
50%,4.000000
75%,6.000000
max,20.000000


## User-book interaction table

This is the core table for the recommender.

For each user-book pair, I compute:

- raw interaction count,
- number of distinct chapters interacted with,
- deepest chapter sequence reached,
- total chapters in that book,
- completion/depth proxy.

I keep both raw frequency and distinct chapter count because repeated interactions may represent stronger engagement, but they could also include repeated logging. Separating both avoids overclaiming.


In [5]:
interactions_with_chapters = interactions.merge(
    chapters[["chapter_id", "book_id", "chapter_sequence_no"]],
    on=["chapter_id", "book_id"],
    how="left"
)

user_book = (
    interactions_with_chapters
    .groupby(["user_id", "book_id"])
    .agg(
        interaction_count=("chapter_id", "size"),
        distinct_chapters=("chapter_id", "nunique"),
        max_chapter_sequence=("chapter_sequence_no", "max"),
        min_chapter_sequence=("chapter_sequence_no", "min")
    )
    .reset_index()
    .merge(book_meta[["book_id", "total_chapters", "author_id", "tags", "tag_list"]], on="book_id", how="left")
)

user_book["chapter_depth_ratio"] = user_book["max_chapter_sequence"] / user_book["total_chapters"]
user_book["chapter_coverage_ratio"] = user_book["distinct_chapters"] / user_book["total_chapters"]

# Engagement weight: balances repeat interactions and depth without letting raw frequency dominate too much.
user_book["engagement_weight"] = (
    np.log1p(user_book["interaction_count"]) 
    + user_book["chapter_depth_ratio"].fillna(0)
    + user_book["chapter_coverage_ratio"].fillna(0)
)

display(user_book.head())
display(user_book[["interaction_count", "distinct_chapters", "chapter_depth_ratio", "chapter_coverage_ratio", "engagement_weight"]].describe())


,user_id,book_id,interaction_count,distinct_chapters,max_chapter_sequence,min_chapter_sequence,total_chapters,author_id,tags,tag_list,chapter_depth_ratio,chapter_coverage_ratio,engagement_weight
0,user_0000013,200976,1,1,5,5,8,97819,Adventure|Fantasy|Graphic Novel|Historical Fic...,"[Adventure, Fantasy, Graphic Novel, Historical...",0.625000,0.125000,1.443147
1,user_0000013,461083,1,1,1,1,8,34356,Adventure|Crime|Dystopian|Fantasy|Graphic Nove...,"[Adventure, Crime, Dystopian, Fantasy, Graphic...",0.125000,0.125000,0.943147
2,user_0000013,469844,1,1,2,2,2,52899,Adventure|Graphic Novel|Humor|Literary Fiction...,"[Adventure, Graphic Novel, Humor, Literary Fic...",1.000000,0.500000,2.193147
3,user_0000013,635456,1,1,7,7,8,46822,Crime|Dystopian|Historical Fiction|Humor|Liter...,"[Crime, Dystopian, Historical Fiction, Humor, ...",0.875000,0.125000,1.693147
4,user_0000013,878246,1,1,1,1,9,22675,Adventure|Crime|Dystopian|Fantasy|Graphic Nove...,"[Adventure, Crime, Dystopian, Fantasy, Graphic...",0.111111,0.111111,0.915369


,interaction_count,distinct_chapters,chapter_depth_ratio,chapter_coverage_ratio,engagement_weight
count,999520.000000,999520.000000,999520.000000,999520.000000,999520.000000
mean,1.000480,1.000480,0.595830,0.192802,1.481974
std,0.021909,0.021909,0.288478,0.161164,0.367830
min,1.000000,1.000000,0.050000,0.050000,0.793147
25%,1.000000,1.000000,0.333333,0.083333,1.193147
50%,1.000000,1.000000,0.600000,0.142857,1.443147
75%,1.000000,1.000000,0.833333,0.250000,1.693147
max,2.000000,2.000000,1.000000,1.000000,3.098612


## Check how much progression signal exists

The assignment highlights chapter progression, but the data decides how heavily we should use it.

Here I check how many user-book pairs have more than one interaction or more than one distinct chapter. This tells us whether a pure sequential next-chapter recommender is strongly supported.


In [6]:
progression_summary = {
    "total_interactions": len(interactions),
    "unique_user_book_pairs": len(user_book),
    "pairs_with_multiple_interactions": int((user_book["interaction_count"] > 1).sum()),
    "pairs_with_multiple_distinct_chapters": int((user_book["distinct_chapters"] > 1).sum()),
    "share_pairs_multiple_interactions": float((user_book["interaction_count"] > 1).mean()),
    "share_pairs_multiple_distinct_chapters": float((user_book["distinct_chapters"] > 1).mean()),
}

display(pd.Series(progression_summary))

user_book["interaction_count"].value_counts().head(10).rename("user_book_pair_count").to_frame()


total_interactions                        1000000.00000
unique_user_book_pairs                     999520.00000
pairs_with_multiple_interactions              480.00000
pairs_with_multiple_distinct_chapters         480.00000
share_pairs_multiple_interactions               0.00048
share_pairs_multiple_distinct_chapters          0.00048
dtype: float64

,user_book_pair_count
interaction_count,
1,999040
2,480


### Interpretation

If only a small share of user-book pairs have multiple chapter interactions, then I should not build the entire solution around next-chapter prediction.

Instead, I use chapter order and chapter depth as preference-strength signals inside a book-level recommendation system.


## Reader profiles

A useful recommender should not behave the same way for every user.

I create reader-level features:

- total interactions,
- number of books interacted with,
- number of authors interacted with,
- repeated user-book engagement,
- average chapter depth,
- tag diversity,
- dominant tag share.

These features help separate casual readers, invested readers, category-focused readers, and generalists.


In [7]:
# User-level aggregate features
user_profile = (
    user_book
    .groupby("user_id")
    .agg(
        total_interactions=("interaction_count", "sum"),
        books_read=("book_id", "nunique"),
        authors_read=("author_id", "nunique"),
        avg_interactions_per_book=("interaction_count", "mean"),
        max_interactions_on_one_book=("interaction_count", "max"),
        avg_chapter_depth=("chapter_depth_ratio", "mean"),
        avg_chapter_coverage=("chapter_coverage_ratio", "mean"),
        total_engagement_weight=("engagement_weight", "sum")
    )
    .reset_index()
)

# Build weighted user-tag profiles
rows = []
for row in user_book[["user_id", "book_id", "tag_list", "engagement_weight"]].itertuples(index=False):
    for tag in row.tag_list:
        rows.append((row.user_id, tag, row.engagement_weight))

user_tag = pd.DataFrame(rows, columns=["user_id", "tag", "tag_weight"])

user_tag_profile = (
    user_tag
    .groupby(["user_id", "tag"], as_index=False)["tag_weight"]
    .sum()
)

user_tag_totals = user_tag_profile.groupby("user_id")["tag_weight"].sum().rename("total_tag_weight")
user_tag_profile = user_tag_profile.merge(user_tag_totals, on="user_id", how="left")
user_tag_profile["tag_share"] = user_tag_profile["tag_weight"] / user_tag_profile["total_tag_weight"]

tag_diversity = (
    user_tag_profile
    .groupby("user_id")
    .agg(
        tag_count=("tag", "nunique"),
        dominant_tag_share=("tag_share", "max")
    )
    .reset_index()
)

user_profile = user_profile.merge(tag_diversity, on="user_id", how="left")
user_profile[["tag_count", "dominant_tag_share"]] = user_profile[["tag_count", "dominant_tag_share"]].fillna(0)

display(user_profile.head())
display(user_profile.describe())


,user_id,total_interactions,books_read,authors_read,avg_interactions_per_book,max_interactions_on_one_book,avg_chapter_depth,avg_chapter_coverage,total_engagement_weight,tag_count,dominant_tag_share
0,user_0000013,7,7,7,1.0,1,0.547736,0.196895,10.064449,15,0.091574
1,user_0000115,6,6,6,1.0,1,0.626852,0.141667,8.769994,15,0.105049
2,user_0000177,6,6,6,1.0,1,0.538478,0.160115,8.350442,15,0.117763
3,user_0000188,8,8,8,1.0,1,0.623016,0.221230,12.299146,15,0.093301
4,user_0000257,5,5,5,1.0,1,0.673183,0.150209,7.582695,15,0.099425


,total_interactions,books_read,authors_read,avg_interactions_per_book,max_interactions_on_one_book,avg_chapter_depth,avg_chapter_coverage,total_engagement_weight,tag_count,dominant_tag_share
count,149803.000000,149803.000000,149803.000000,149803.000000,149803.000000,149803.000000,149803.000000,149803.000000,149803.000000,149803.000000
mean,6.675434,6.672230,6.667630,1.000490,1.003198,0.595617,0.192745,9.888071,14.705520,0.106890
std,2.572317,2.570475,2.567994,0.009834,0.056456,0.123381,0.068718,3.929454,1.032073,0.021808
min,1.000000,1.000000,1.000000,1.000000,1.000000,0.050000,0.050000,0.793147,1.000000,0.066667
25%,5.000000,5.000000,5.000000,1.000000,1.000000,0.516667,0.146958,7.086630,15.000000,0.094129
50%,6.000000,6.000000,6.000000,1.000000,1.000000,0.596108,0.180556,9.602743,15.000000,0.102766
75%,8.000000,8.000000,8.000000,1.000000,1.000000,0.674837,0.224628,12.378511,15.000000,0.114365
max,20.000000,20.000000,20.000000,2.000000,2.000000,1.000000,1.000000,32.835409,15.000000,1.000000


In [8]:
def assign_reader_segment(row):
    if row["books_read"] <= 2 or row["total_interactions"] <= 2:
        return "casual_reader"
    if row["max_interactions_on_one_book"] > 1 or row["avg_chapter_depth"] >= 0.60:
        if row["dominant_tag_share"] >= 0.45:
            return "invested_category_reader"
        return "invested_reader"
    if row["dominant_tag_share"] >= 0.45 and row["books_read"] >= 3:
        return "category_focused_reader"
    if row["tag_count"] >= 6 and row["dominant_tag_share"] < 0.35:
        return "generalist_reader"
    return "standard_reader"

user_profile["reader_segment"] = user_profile.apply(assign_reader_segment, axis=1)

segment_counts = (
    user_profile["reader_segment"]
    .value_counts()
    .rename_axis("reader_segment")
    .reset_index(name="users")
)
segment_counts["share"] = segment_counts["users"] / segment_counts["users"].sum()

display(segment_counts)


,reader_segment,users,share
0,generalist_reader,73828,0.492834
1,invested_reader,70444,0.470244
2,casual_reader,5530,0.036915
3,standard_reader,1,0.000007


## Book profiles

I also profile books so the model can distinguish:

- globally popular books,
- books trending inside categories,
- books with stronger engagement-quality signals,
- books with high chapter-level attention.


In [9]:
book_profile = (
    interactions_with_chapters
    .groupby("book_id")
    .agg(
        total_interactions=("chapter_id", "size"),
        unique_users=("user_id", "nunique"),
        unique_chapters_interacted=("chapter_id", "nunique"),
        avg_chapter_sequence_interacted=("chapter_sequence_no", "mean"),
        max_chapter_sequence_interacted=("chapter_sequence_no", "max")
    )
    .reset_index()
    .merge(book_meta, on="book_id", how="left")
)

book_profile["interaction_per_user"] = book_profile["total_interactions"] / book_profile["unique_users"]
book_profile["avg_depth_signal"] = book_profile["avg_chapter_sequence_interacted"] / book_profile["total_chapters"]
book_profile["chapter_reach_ratio"] = book_profile["unique_chapters_interacted"] / book_profile["total_chapters"]

# Popularity categories based on unique users
book_profile["book_popularity_segment"] = pd.cut(
    book_profile["unique_users"],
    bins=[-1, 10, 50, 200, np.inf],
    labels=["niche", "emerging", "popular", "very_popular"]
)

display(book_profile.head())
display(book_profile[[
    "total_interactions", "unique_users", "interaction_per_user",
    "avg_depth_signal", "chapter_reach_ratio"
]].describe())

display(book_profile["book_popularity_segment"].value_counts().rename("books").to_frame())


,book_id,total_interactions,unique_users,unique_chapters_interacted,avg_chapter_sequence_interacted,max_chapter_sequence_interacted,total_chapters,author_id,first_published_date,tags,tag_list,interaction_per_user,avg_depth_signal,chapter_reach_ratio,book_popularity_segment
0,100089,41,41,3,2.146341,3,3,55121,1995-03-24,Fantasy|Science Fiction,"[Fantasy, Science Fiction]",1.0,0.715447,1.0,emerging
1,100096,17,17,1,1.000000,1,1,61876,2015-12-31,Romance,[Romance],1.0,1.000000,1.0,emerging
2,100193,42,42,4,2.357143,4,4,67983,1996-01-26,Crime|Dystopian|Fantasy|Humor|Literary Fiction...,"[Crime, Dystopian, Fantasy, Humor, Literary Fi...",1.0,0.589286,1.0,emerging
3,100205,233,233,4,1.339056,4,4,64099,2016-02-26,Adventure|Fantasy|Horror|Literary Fiction|Myst...,"[Adventure, Fantasy, Horror, Literary Fiction,...",1.0,0.334764,1.0,very_popular
4,100301,31,31,3,1.967742,3,3,77538,2015-08-14,Dystopian|Humor,"[Dystopian, Humor]",1.0,0.655914,1.0,emerging


,total_interactions,unique_users,interaction_per_user,avg_depth_signal,chapter_reach_ratio
count,9575.000000,9575.000000,9575.000000,9575.000000,9575.000000
mean,104.438642,104.388512,1.000190,0.662212,0.999973
std,123.832902,123.705961,0.001315,0.161935,0.002121
min,2.000000,2.000000,1.000000,0.232123,0.800000
25%,28.000000,28.000000,1.000000,0.566146,1.000000
50%,49.000000,49.000000,1.000000,0.632909,1.000000
75%,140.500000,140.500000,1.000000,0.733333,1.000000
max,1166.000000,1161.000000,1.035714,1.000000,1.000000


,books
book_popularity_segment,
emerging,4500
popular,2537
very_popular,2084
niche,454


## Tag/category trends

Tags are important for category-focused readers and for cold-start-style recommendations.

I look at both metadata-level tag distribution and interaction-weighted tag distribution.


In [10]:
# Metadata-level tag counts
chapter_tags = (
    chapters[["chapter_id", "book_id", "tags"]]
    .assign(tag=lambda df: df["tags"].astype(str).str.split("|"))
    .explode("tag")
)
chapter_tags["tag"] = chapter_tags["tag"].str.strip()

metadata_tag_counts = (
    chapter_tags["tag"]
    .value_counts()
    .rename_axis("tag")
    .reset_index(name="chapter_count")
)

# Interaction-weighted tag counts
interaction_tags = interactions.merge(
    chapter_tags[["chapter_id", "book_id", "tag"]],
    on=["chapter_id", "book_id"],
    how="left"
)

interaction_tag_counts = (
    interaction_tags["tag"]
    .value_counts()
    .rename_axis("tag")
    .reset_index(name="interaction_count")
)

tag_trends = metadata_tag_counts.merge(interaction_tag_counts, on="tag", how="outer").fillna(0)
tag_trends["interaction_share"] = tag_trends["interaction_count"] / tag_trends["interaction_count"].sum()
tag_trends = tag_trends.sort_values("interaction_count", ascending=False)

display(tag_trends)


,tag,chapter_count,interaction_count,interaction_share
14,Young Adult,8393,173178,0.069085
1,Crime,8589,170275,0.067927
5,Historical Fiction,8547,169958,0.067801
13,Thriller,8325,168582,0.067252
2,Dystopian,8448,168551,0.067240
4,Graphic Novel,8376,168550,0.067239
9,Mystery,8203,168243,0.067117
6,Horror,8235,168204,0.067101
11,Romance,8389,167290,0.066737
10,Paranormal,8359,166681,0.066494


## Chapter-level trends

Chapter popularity helps with continuation/revisit actions.

Even though the main output is book-level, chapter-level trends can tell us which books or chapters attract attention and whether later chapters receive meaningful engagement.


In [11]:
chapter_profile = (
    interactions_with_chapters
    .groupby(["book_id", "chapter_id", "chapter_sequence_no"])
    .agg(
        chapter_total_interactions=("user_id", "size"),
        chapter_unique_users=("user_id", "nunique")
    )
    .reset_index()
    .merge(book_meta[["book_id", "total_chapters", "author_id", "tags"]], on="book_id", how="left")
)

chapter_profile["chapter_depth_ratio"] = chapter_profile["chapter_sequence_no"] / chapter_profile["total_chapters"]

display(chapter_profile.head())
display(chapter_profile[["chapter_total_interactions", "chapter_unique_users", "chapter_depth_ratio"]].describe())

sequence_trend = (
    chapter_profile
    .groupby("chapter_sequence_no")
    .agg(
        total_interactions=("chapter_total_interactions", "sum"),
        unique_chapters=("chapter_id", "nunique"),
        avg_interactions_per_chapter=("chapter_total_interactions", "mean")
    )
    .reset_index()
)

display(sequence_trend.head(20))


,book_id,chapter_id,chapter_sequence_no,chapter_total_interactions,chapter_unique_users,total_chapters,author_id,tags,chapter_depth_ratio
0,100089,3515069,2,15,15,3,55121,Fantasy|Science Fiction,0.666667
1,100089,5011702,1,10,10,3,55121,Fantasy|Science Fiction,0.333333
2,100089,6062471,3,16,16,3,55121,Fantasy|Science Fiction,1.000000
3,100096,1727513,1,17,17,1,61876,Romance,1.000000
4,100193,1591522,2,10,10,4,67983,Crime|Dystopian|Fantasy|Humor|Literary Fiction...,0.500000


,chapter_total_interactions,chapter_unique_users,chapter_depth_ratio
count,49998.000000,49998.000000,49998.000000
mean,20.000800,20.000800,0.595755
std,41.567305,41.567305,0.290739
min,1.000000,1.000000,0.050000
25%,8.000000,8.000000,0.333333
50%,11.000000,11.000000,0.600000
75%,13.000000,13.000000,0.857143
max,264.000000,264.000000,1.000000


,chapter_sequence_no,total_interactions,unique_chapters,avg_interactions_per_chapter
0,1,192094,9575,20.062037
1,2,169096,8630,19.593975
2,3,145031,7304,19.856380
3,4,116760,5677,20.567201
4,5,89208,4216,21.159393
5,6,63571,3064,20.747715
6,7,45782,2347,19.506604
7,8,33565,1744,19.245986
8,9,26766,1387,19.297765
9,10,20635,1086,19.000921


## Modeling direction from the analysis

Based on the data, I would build a hybrid recommender that uses reader and book profiles rather than only item popularity.

### Candidate recommendation types

1. **Explore new book**  
   For most users, recommend books not already in their history using collaborative similarity, genre affinity, author affinity, and popularity.

2. **Revisit partially explored book**  
   For users with partial book engagement, identify books worth revisiting when the book has strong engagement-quality signals.

3. **Continue chapter**  
   If the user has clear multi-chapter progression in a book, recommend the next chapter. But because progression is sparse, this should not be the only recommendation type.

### Main scoring signals

For a candidate book:

- collaborative score: users/books with similar interaction patterns,
- genre affinity score: match with weighted user tag profile,
- author affinity score: match with authors the user engaged with,
- engagement-quality score: book has later-chapter or repeat-engagement signals,
- popularity score: fallback and tie-breaker.

The final system should adapt by reader segment:

- casual readers: more popularity/category trending,
- category-focused readers: more genre affinity and category-trending books,
- invested readers: more collaborative and author/depth signals,
- generalists: broader candidate pool and less narrow genre matching.


## Next implementation step

The next notebook section should implement the actual recommender:

1. Build candidate books for a user.
2. Compute genre affinity.
3. Compute author affinity.
4. Compute popularity score.
5. Add collaborative similarity using a user-book sparse matrix.
6. Combine scores into a final rank.
7. Evaluate using held-out user-book interactions.

A good evaluation setup would hold out one book per user and test whether the recommender can recover it in the top-K recommendations.
